# Movies Recommender System

In this project, we build and evaluate movie recommenders on Kaggle's [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset): TMDB metadata for ~45k movies plus a 100k-rating MovieLens sample.

Files expected in `DATA_DIR`: `movies_metadata.csv`, `credits.csv`, `keywords.csv`, `ratings_small.csv`, `links_small.csv`.

We build:
1. A non-personalised **popularity chart** (IMDB weighted rating)
2. Two **content-based** recommenders (plot text with TF-IDF; cast/director/genres/keywords)
3. **Collaborative filtering** (item-item and SVD matrix factorisation)
4. A simple **hybrid** (reciprocal rank fusion)

and compare them with a time-based **offline evaluation** (Precision/Recall/NDCG@10, RMSE).

In [ ]:
from pathlib import Path
from ast import literal_eval
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize
from nltk.stem.snowball import SnowballStemmer
from surprise import Dataset, Reader, SVD, BaselineOnly, accuracy

DATA_DIR = Path('.')
RANDOM_STATE = 42

In [ ]:
md = pd.read_csv(DATA_DIR / 'movies_metadata.csv', low_memory=False)
md.shape

In [ ]:
# Check for missing values
md.isnull().sum()

#### Cleaning
* A few rows have malformed ids (dates in the `id` column) and some movies appear twice with the same id: drop both.
* Titles are **not** unique (remakes, re-releases), so we keep every movie and resolve title lookups to the most-voted version later.

In [ ]:
md['id'] = pd.to_numeric(md['id'], errors='coerce')
md = md.dropna(subset=['id'])
md['id'] = md['id'].astype(int)
md = md.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

md['genres'] = md['genres'].fillna('[]').apply(literal_eval).apply(lambda gs: [g['name'] for g in gs])
md['release_date'] = pd.to_datetime(md['release_date'], errors='coerce')
md['year'] = md['release_date'].dt.year.astype('Int64')
md['vote_count'] = pd.to_numeric(md['vote_count'], errors='coerce')
md['vote_average'] = pd.to_numeric(md['vote_average'], errors='coerce')
md.shape

## Popularity chart

We use IMDB's *weighted rating* formula to construct the chart:

Weighted Rating (WR) = $(\frac{v}{v + m} \cdot R) + (\frac{m}{v + m} \cdot C)$

where,
* *v* is the number of votes for the movie
* *m* is the minimum votes required to be listed in the chart
* *R* is the average rating of the movie
* *C* is the mean vote across the whole pool

It is a Bayesian average: movies with few votes are shrunk towards the mean *C*, so a 10/10 from 3 votes cannot top the chart. We use the **95th percentile** of vote counts as *m*. Both *m* and *C* are computed over movies with at least one vote (unrated movies have `vote_average = 0`, which would drag *C* down), and within the genre pool when building genre charts.

In [ ]:
def weighted_rating(v, R, m, C):
    return (v / (v + m)) * R + (m / (v + m)) * C


def build_chart(df, genres=None, percentile=0.95, top_n=250):
    # Top-N movies by weighted rating; m and C are computed within the (genre-filtered) pool
    if genres:
        df = df[df['genres'].apply(lambda gs: all(g in gs for g in genres))]
    rated = df[df['vote_count'] > 0]
    C = rated['vote_average'].mean()
    m = rated['vote_count'].quantile(percentile)
    qualified = rated[rated['vote_count'] >= m].copy()
    qualified['score'] = weighted_rating(qualified['vote_count'], qualified['vote_average'], m, C)
    print(f'C = {C:.3f}, m = {m:.0f}, qualified movies = {len(qualified):,}')
    cols = ['title', 'year', 'vote_count', 'vote_average', 'score']
    return qualified.sort_values('score', ascending=False)[cols].head(top_n)

#### Top 10 movies

In [ ]:
build_chart(md).head(10)

## Create chart based on genre(s)

In [ ]:
build_chart(md, ['Action', 'Romance']).head(10)

In [ ]:
build_chart(md, ['Romance']).head(10)

## Content Based Recommender

The chart gives the same recommendations to everyone. To personalise, we compute similarity between movies from their content and suggest movies similar to one the user liked (**Content Based Filtering**).

We build two content-based recommenders based on:
* Movie Overviews and Taglines
* Movie Cast, Director, Genres and Keywords

Both return the nearest neighbours by cosine similarity. Instead of materialising the full N x N similarity matrix (45k x 45k float64 is ~16 GB), we compute similarities for the query row only, which needs O(N) memory per query.

In [ ]:
# Titles are not unique: map each title to the row of its most-voted version
title_to_idx = (md.sort_values('vote_count', ascending=False)
                  .drop_duplicates('title')
                  .reset_index()
                  .set_index('title')['index'])


def recommend_movie(title, features, top_n=10):
    # features: one L2-normalised row per movie in md, so a dot product is the cosine similarity
    if title not in title_to_idx.index:
        raise KeyError(f'Movie not found: {title!r}')
    idx = title_to_idx[title]
    sims = linear_kernel(features[idx], features).ravel()
    sims[idx] = -np.inf  # skip the movie itself
    top = np.argpartition(-sims, top_n)[:top_n]
    top = top[np.argsort(-sims[top])]
    return md.iloc[top][['title', 'year']].assign(similarity=sims[top].round(3))

##### 1. Recommendation using Movie Overviews and Taglines

In [ ]:
def preprocess(text):
    # A custom preprocessor replaces scikit-learn's default one, so lowercasing has to happen here
    # (lowercase=True is ignored when preprocessor is set). Numbers become a single placeholder token.
    return re.sub(r'\d+', ' num ', text.lower())


tfidf = TfidfVectorizer(min_df=2, max_df=0.85, stop_words='english', preprocessor=preprocess)

md['description'] = md['overview'].fillna('') + ' ' + md['tagline'].fillna('')
tfidf_matrix = tfidf.fit_transform(md['description'])  # rows are L2-normalised
tfidf_matrix.shape

In [ ]:
recommend_movie("The Godfather", tfidf_matrix)

In [ ]:
recommend_movie("The Dark Knight", tfidf_matrix)

In [ ]:
recommend_movie("The Avengers", tfidf_matrix)

The description recommender mostly finds movies from the same franchise (Batman films for **The Dark Knight**). Someone who liked **The Dark Knight** may like it because of Nolan, so next we use more suggestive metadata than **Overview** and **Tagline**.

##### 2. Metadata Based Recommender

In [ ]:
credits = pd.read_csv(DATA_DIR / 'credits.csv').drop_duplicates('id')
keywords = pd.read_csv(DATA_DIR / 'keywords.csv').drop_duplicates('id')

# Join on the TMDB id column. (DataFrame.join(on='id') would match md.id against the *row index*
# of credits and attach another movie's cast and crew.)
n_movies = len(md)
md = md.merge(credits, on='id', how='left').merge(keywords, on='id', how='left')
assert len(md) == n_movies, 'merge must not duplicate movies'

In [ ]:
for col in ['cast', 'crew', 'keywords']:
    md[col] = md[col].fillna('[]').apply(literal_eval)


def get_director(crew):
    return next((c['name'] for c in crew if c.get('job') == 'Director'), None)


md['director'] = md['crew'].apply(get_director)
md['cast'] = md['cast'].apply(lambda xs: [x['name'] for x in xs[:3]])
md['keywords'] = md['keywords'].apply(lambda xs: [x['name'] for x in xs])

In [ ]:
# Sanity check on a known movie
md.loc[title_to_idx['The Godfather'], ['title', 'director', 'cast']]

We create a metadata "soup" for every movie of **top-3 cast, director, genres and keywords**, vectorise it with a **Count Vectorizer** and compare movies by cosine similarity.

1. **Strip spaces and lowercase** every name, so that **Johnny Depp** and **Johnny Galecki** do not share the token *johnny*.
2. **Prefix each token with its field** (`cast:`, `director:`, `genre:`, `keyword:`) so values from different fields never collide.
3. **Mention the director 3 times** to give it more weight relative to the cast.
4. **Keywords** that appear only once in the dataset are dropped, and the rest are stemmed so *dogs* and *dog* match.

In [ ]:
keyword_counts = md['keywords'].explode().value_counts()
frequent_keywords = set(keyword_counts[keyword_counts > 1].index)
stemmer = SnowballStemmer('english')
md['keywords'] = md['keywords'].apply(lambda ks: [stemmer.stem(k) for k in ks if k in frequent_keywords])

DIRECTOR_WEIGHT = 3


def clean_token(name):
    return name.lower().replace(' ', '')


def metadata_tokens(row):
    tokens = [f'cast:{clean_token(c)}' for c in row['cast']]
    if isinstance(row['director'], str):
        tokens += [f'director:{clean_token(row["director"])}'] * DIRECTOR_WEIGHT
    tokens += [f'genre:{clean_token(g)}' for g in row['genres']]
    tokens += [f'keyword:{clean_token(k)}' for k in row['keywords']]
    return tokens


def identity(tokens):
    return tokens


md['metadata_tokens'] = md.apply(metadata_tokens, axis=1)
count = CountVectorizer(analyzer=identity)  # tokens are already prepared
count_matrix = normalize(count.fit_transform(md['metadata_tokens']))
count_matrix.shape

In [ ]:
recommend_movie('The Dark Knight', count_matrix)

In [ ]:
recommend_movie('The Godfather', count_matrix)

## Collaborative Filtering

Content-based engines only suggest movies *close* to a given movie and ignore what similar users liked. **Collaborative Filtering** uses the ratings of other users instead. We build two:

* **Item-item CF**: two movies are similar if the same users rate them similarly. Ratings are mean-centred per user (so generous and harsh raters are comparable); the similarity is the cosine of the centred rating columns.
* **SVD** (matrix factorisation, [Surprise](https://surprise.readthedocs.io/)): $\hat r_{ui} = \mu + b_u + b_i + q_i^\top p_u$, learned by SGD with L2 regularisation.

`ratings_small.csv` uses MovieLens `movieId`s; `links_small.csv` maps them to TMDB ids, i.e. to rows of `md`, so all models share one item space.

In [ ]:
ratings = pd.read_csv(DATA_DIR / 'ratings_small.csv')
links = pd.read_csv(DATA_DIR / 'links_small.csv')

links = links.dropna(subset=['tmdbId']).astype({'tmdbId': int})
tmdb_to_row = pd.Series(md.index, index=md['id'])
links['row'] = links['tmdbId'].map(tmdb_to_row)
links = links.dropna(subset=['row']).astype({'row': int}).drop_duplicates('movieId')

n_raw = len(ratings)
ratings = (ratings.merge(links[['movieId', 'row']], on='movieId', how='inner')
                  .sort_values(['userId', 'timestamp'])
                  .drop_duplicates(['userId', 'row'], keep='last'))
print(f'{len(ratings):,} of {n_raw:,} ratings mapped to TMDB movies; '
      f'{ratings.userId.nunique()} users, {ratings.row.nunique():,} movies')

## Offline evaluation

**Split.** For every user, the most recent 20% of their ratings (by timestamp) go to the test set and the rest to training. A random split would let models learn from a user's future ratings.

**Relevance.** A test movie rated **4.0 or higher** is relevant.

**Task.** For each user, rank all movies in the catalogue they have not rated in training, and take the top **K = 10**.

**Metrics.**
* *Precision@10*, *Recall@10*, *Hit rate@10*, and *NDCG@10* (rewards relevant movies near the top), averaged over users with at least one relevant test movie, with a 95% bootstrap confidence interval for NDCG.
* *Coverage*: share of the catalogue that appears in anyone's top 10.
* *Avg. popularity*: mean number of training ratings of recommended movies (popularity bias).
* *RMSE/MAE* on all test ratings for the rating-prediction models.

Caveats: the catalogue is the set of movies rated by someone in `ratings_small`, and the split is per user, so a test rating of one user can be older than a training rating of another.

In [ ]:
TEST_FRACTION = 0.2
MIN_RATINGS = 5
RELEVANCE_THRESHOLD = 4.0
K = 10

recency_rank = ratings.groupby('userId').cumcount(ascending=False)  # 0 = most recent rating
n_user_ratings = ratings.groupby('userId')['rating'].transform('size')
is_test = (recency_rank < np.floor(n_user_ratings * TEST_FRACTION)) & (n_user_ratings >= MIN_RATINGS)
train, test = ratings[~is_test], ratings[is_test]

users = np.sort(ratings['userId'].unique())
items = np.sort(ratings['row'].unique())  # catalogue: rows of md
user_pos = pd.Series(np.arange(len(users)), index=users)
item_pos = pd.Series(np.arange(len(items)), index=items)
n_users, n_items = len(users), len(items)


def to_matrix(df, values):
    return sparse.csr_matrix(
        (values, (user_pos[df['userId']].to_numpy(), item_pos[df['row']].to_numpy())),
        shape=(n_users, n_items))


user_mean = train.groupby('userId')['rating'].mean()
R_train = to_matrix(train, train['rating'].to_numpy())
R_centered = to_matrix(train, (train['rating'] - train['userId'].map(user_mean)).to_numpy())
seen = to_matrix(train, np.ones(len(train)))
item_train_count = np.asarray(seen.sum(axis=0)).ravel()

relevant_test = test[test['rating'] >= RELEVANCE_THRESHOLD]
relevant = {user_pos[u]: set(item_pos[g['row']]) for u, g in relevant_test.groupby('userId')}

print(f'train: {len(train):,} ratings, test: {len(test):,} ratings, '
      f'evaluated users: {len(relevant)}, catalogue: {n_items:,} movies')

In [ ]:
def mask_seen(scores):
    scores = np.array(scores, dtype=float)
    scores[seen.nonzero()] = -np.inf
    return scores


def top_k(scores, k=K):
    scores = mask_seen(scores)
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, top, axis=1), axis=1)
    return np.take_along_axis(top, order, axis=1)


def evaluate(scores, k=K):
    top = top_k(scores, k)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    rows = []
    for u, rel in relevant.items():
        hits = np.isin(top[u], list(rel))
        ideal = discounts[:min(len(rel), k)].sum()
        rows.append({'precision': hits.sum() / k, 'recall': hits.sum() / len(rel),
                     'ndcg': (hits * discounts).sum() / ideal, 'hit_rate': float(hits.any())})
    per_user = pd.DataFrame(rows)
    recommended = top[list(relevant)]
    summary = per_user.mean().to_dict()
    summary['coverage'] = len(np.unique(recommended)) / n_items
    summary['avg_popularity'] = item_train_count[recommended].mean()
    return summary, per_user


def bootstrap_ci(values, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_boot, len(values))).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

### Models

All models produce a score for every (user, movie) pair; movies the user rated in training are excluded before ranking.

* **Most popular**: number of training ratings (same list for everyone).
* **Weighted rating**: the IMDB formula on training ratings, *m* = 90th percentile of training rating counts.
* **Content (TF-IDF / metadata)**: a user profile is the sum of the feature vectors of the movies they rated, weighted by their mean-centred rating (liked movies pull towards, disliked push away). Score = profile . movie vector.
* **Item-item CF**: score(u, i) = sum over rated j of sim(i, j) x centred rating(u, j).
* **SVD**: Surprise defaults (100 factors, 20 epochs, lr 0.005, reg 0.02), not tuned.
* **Hybrid**: reciprocal rank fusion of SVD and metadata content, $\frac{1}{60 + rank_{SVD}} + \frac{1}{60 + rank_{content}}$. RRF needs no score calibration and has no weights fitted on the test set.

In [ ]:
scores = {}

# Popularity baselines
scores['Most popular'] = np.broadcast_to(item_train_count, (n_users, n_items))

item_sum = np.asarray(R_train.sum(axis=0)).ravel()
item_avg = np.divide(item_sum, item_train_count, out=np.zeros(n_items), where=item_train_count > 0)
m_train = np.quantile(item_train_count[item_train_count > 0], 0.90)
wr = weighted_rating(item_train_count, item_avg, m_train, train['rating'].mean())
scores['Weighted rating'] = np.broadcast_to(wr, (n_users, n_items))


# Content-based: rating-weighted user profiles in TF-IDF / metadata space
def content_scores(features):
    X = features[items]
    profiles = R_centered @ X
    return (profiles @ X.T).toarray()


scores['Content: TF-IDF'] = content_scores(tfidf_matrix)
scores['Content: metadata'] = content_scores(count_matrix)

# Item-item CF: R_c @ S with S = Xn.T @ Xn, computed without materialising the items x items matrix
Xn = normalize(R_centered, axis=0)
user_user = (R_centered @ Xn.T).toarray()
scores['Item-item CF'] = (Xn.T @ user_user.T).T

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
trainset = Dataset.load_from_df(train[['userId', 'row', 'rating']], reader).build_full_trainset()
testset = list(test[['userId', 'row', 'rating']].itertuples(index=False, name=None))

baseline = BaselineOnly(verbose=False)
svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=RANDOM_STATE)

rating_metrics = {}
for name, algo in [('Baseline (biases only)', baseline), ('SVD', svd)]:
    algo.fit(trainset)
    predictions = algo.test(testset)
    rating_metrics[name] = {'RMSE': accuracy.rmse(predictions, verbose=False),
                            'MAE': accuracy.mae(predictions, verbose=False)}

pd.DataFrame(rating_metrics).T.round(4)

In [ ]:
# Score every (user, movie) pair from the learned SVD parameters; movies unseen in training keep b_i = 0, q_i = 0
bi = np.zeros(n_items)
qi = np.zeros((n_items, svd.n_factors))
for pos, row in enumerate(items):
    try:
        inner = trainset.to_inner_iid(row)
    except ValueError:
        continue
    bi[pos], qi[pos] = svd.bi[inner], svd.qi[inner]
inner_users = [trainset.to_inner_uid(u) for u in users]
bu, pu = svd.bu[inner_users], svd.pu[inner_users]
scores['SVD'] = trainset.global_mean + bu[:, None] + bi[None, :] + pu @ qi.T


def ranks(s):
    return np.argsort(np.argsort(-mask_seen(s), axis=1), axis=1) + 1


RRF_K = 60
scores['Hybrid (SVD + metadata, RRF)'] = (1 / (RRF_K + ranks(scores['SVD']))
                                          + 1 / (RRF_K + ranks(scores['Content: metadata'])))

### Results

In [ ]:
results, ndcg_ci = {}, {}
for name, s in scores.items():
    summary, per_user = evaluate(s)
    results[name] = summary
    ndcg_ci[name] = bootstrap_ci(per_user['ndcg'].to_numpy())

results = pd.DataFrame(results).T
results['ndcg_ci_low'] = [ndcg_ci[n][0] for n in results.index]
results['ndcg_ci_high'] = [ndcg_ci[n][1] for n in results.index]
results = results.sort_values('ndcg', ascending=False)
results.columns = [f'{c}@{K}' if c in ('precision', 'recall', 'ndcg', 'hit_rate') else c for c in results.columns]
results.round(4)

In [ ]:
ndcg = results[f'ndcg@{K}']
errors = [ndcg - results['ndcg_ci_low'], results['ndcg_ci_high'] - ndcg]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ndcg.index[::-1], ndcg[::-1], xerr=[e[::-1] for e in errors], color='steelblue', capsize=3)
ax.set_xlabel(f'NDCG@{K} (95% bootstrap CI)')
ax.set_title('Offline ranking quality, time-based split')
plt.tight_layout()
plt.show()

#### How to read the results

* **Popularity baselines are the bar to beat.** On small, popularity-skewed MovieLens samples they are often hard to beat; a personalised model that does not beat them adds no value.
* **RMSE and ranking disagree.** SVD is trained to minimise squared error on observed ratings; that does not guarantee a good top-10 among *all* unrated movies (most of which the user never would have watched).
* **Coverage and avg. popularity** show the trade-off: popularity models recommend the same few blockbusters to everyone, content models spread across the catalogue.
* Overlapping confidence intervals mean the difference between two models is not reliable at this sample size (~670 users).

Next steps: tune hyperparameters (SVD factors/regularisation, content weights, RRF k) on a validation split carved out of training, try implicit-feedback models (ALS/BPR), and train a learning-to-rank model (e.g. LightGBM LambdaMART) on the model scores plus popularity and recency features.